In [ ]:
import os

# Trick to have TeX figures
os.environ["LD_LIBRARY_PATH"] = ""
os.environ["CONDA_PREFIX"] = "/home/guerrini/.conda/envs/sp_validation_3.11"

from getdist import plots, loadMCSamples
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use(
    "/home/guerrini/matplotlib_config/paper.mplstyle"
)

sns.set_palette("husl")

g = plots.get_subplot_plotter(width_inch=7)
g.settings.axes_fontsize=15
g.settings.axes_labelsize=15
g.settings.alpha_filled_add = 0.7
g.settings.legend_fontsize = 15

%matplotlib inline

#SPECIFY DATA DIRECTORY AND DESIRED CHAINS TO ANALYSE
root_dir = "/n09data/guerrini/glass_mock_chains/glass_mock_v2_00001/glass_mock_v2_00001/"

roots = [
    f"glass_mock_v2_00001_cell",
    f"glass_mock_v2_00001_cell_mh",
    f"glass_mock_v2_00001_cell_no_sys"
]

print(roots)

In [ ]:
# MAKE PARAMNAMES FILE

for root in roots:
    try:
        if root != "glass_mock_v2_00001_cell_mh":
            with open(root_dir + '/samples_{}.txt'.format(root), "r") as file:
                params = file.readline()[1:].split('\t')[:-4]
                file.close()
        else:
            with open(root_dir + '/samples_{}_1.txt'.format(root), "r") as file:
                params = file.readline()[1:].split('\t')[:-2]
                file.close()
    
        with open(root_dir + '/getdist_{}.paramnames'.format(root), "w") as file:
            for i in range(len(params)):
                if len(params[i].split('--')) > 1:
                    file.write(params[i].split('--')[1] + '\n')
                else:
                    file.write(params[i].split('--')[0] + '\n')
            file.close()
    except Exception as e:
        print(e)

In [ ]:
#READ CHAIN

chains=[]
mask_labels = []

for root in roots:

    try:
        if root != "glass_mock_v2_00001_cell_mh":
            samples = np.loadtxt(root_dir + 'samples_{}.txt'.format(root))
            print(len(samples))
            if 'nautilus' in root:
                samples = np.column_stack((np.exp(samples[:,-3]),samples[:,-1]-samples[:,-2],samples[:,0:-3]))
            else:
                samples = np.column_stack((samples[:,-1],samples[:,-2],samples[:,0:-4]))
            np.savetxt(root_dir + 'getdist_{}.txt'.format(root), samples)
            
            chain = g.samples_for_root(root_dir + 'getdist_{}'.format(root),
                                    cache=False,
                                    settings={'ignore_rows':0,
                                                'smooth_scale_2D':0.5,
                                                'smooth_scale_1D':0.5})

            chains.append(chain)
            mask_labels.append(1)
        else:
            for i in range(1,5):
                samples = np.loadtxt(root_dir + 'samples_{}_{}.txt'.format(root, i))
                print(len(samples))
                samples = np.column_stack((samples[:,-1],samples[:,-2],samples[:,0:-2]))
                np.savetxt(root_dir + 'getdist_{}_{}.txt'.format(root, i), samples)
                
            chain = g.samples_for_root(root_dir + 'getdist_{}'.format(root),
                                    cache=False,
                                    settings={'ignore_rows':0,
                                                'smooth_scale_2D':0.3,
                                                'smooth_scale_1D':0.3})

            chains.append(chain)
            mask_labels.append(1)
    except Exception as e:
        print("ERRROOOOOOOOR")
        print(e)
        mask_labels.append(0)

In [ ]:
name_list = ['OMEGA_M','ombh2','h0','n_s','SIGMA_8','s_8_input', 'logt_agn']
label_list = ['\Omega_m', '\omega_b h^2', 'h_0', 'n_s', '\sigma_8', 'S_8', 'log T_{AGN}']

for chain in chains:
    param_names = chain.getParamNames()
    for name, label in zip(name_list, label_list):
        param_names.parWithName(name).label = label

In [ ]:
from astropy.cosmology import Planck18 as planck

Omega_m_fid = planck.Om0
sigma_8_fid = 0.8054
s8_fid = sigma_8_fid * (Omega_m_fid / 0.3)**0.5
h = planck.h
Omega_b_fig = planck.Ob0
n_s_fid = 0.965
print(f"Fiducial values: Omega_m = {Omega_m_fid}, sigma_8 = {sigma_8_fid}, S_8 = {s8_fid}")

markers = {
    "OMEGA_M": Omega_m_fid,
    "SIGMA_8": sigma_8_fid,
    "s_8_input": s8_fid,
    "h0": h,
    "ombh2": Omega_b_fig * h**2,
    "n_s": n_s_fid
}

In [ ]:
legend_labels = np.array([
    "GLASS mock fiducial",
    "GLASS mock with MH",
    "GLASS mock no sys"
])

legend_labels = legend_labels[np.array(mask_labels, dtype=bool)]


""" #Plot all parameters
g.triangle_plot(
    chains,
    ['OMEGA_M','ombh2', 'h0','n_s','SIGMA_8','s_8_input', 'logt_agn','a','m1','bias_1'],
    legend_labels=legend_labels,
    legend_loc='upper right',
    filled=True,
    markers=markers
)

plt.show() """

In [ ]:
#Plot only cosmological parameters

g.triangle_plot(
    chains,
    ['OMEGA_M','s_8_input', 'SIGMA_8', 'a'],
    legend_labels=legend_labels,
    legend_loc='upper right',
    filled=True,
    markers=markers
)


plt.show()

In [ ]:
preliminary_watermark = False
#Plot S8 Omega_m only
g.triangle_plot(
    chains,
    ['OMEGA_M','s_8_input'],
    legend_labels=legend_labels,
    legend_loc='upper right',
    filled=True,
    title_limit=1,
    markers=markers
)

if preliminary_watermark:
    plt.figtext(0.5, 0.5, 'PRELIMINARY',
            fontsize=150, color='gray',
            ha='center', va='center',
            alpha=0.3, rotation=330)

plt.show()

In [ ]:
len(chains)

In [ ]:
for i, chain in enumerate(chains):
    print(legend_labels[i])
    samples = chain.samples

    likestats = chain.getLikeStats()
    margestats = chain.getMargeStats()

    param_name = 'S_8'
    print(param_name)
    param_names = chain.getParamNames()
    par = param_names.parWithName(param_name)
    param_stats = margestats.parWithName(param_name)
    print("Average:", param_stats.mean)

    idx_map = np.argmax(chain.loglikes)
    s8_index_in_samples = chain.index[param_name]
    S8_map = samples[idx_map, s8_index_in_samples]
    print("MAP:", S8_map)

    S8_maxlike = samples[-1, s8_index_in_samples]
    print("Max Likelihood:", S8_maxlike)

    samples = chain.samples

    likestats = chain.getLikeStats()
    margestats = chain.getMargeStats()

    param_name = 'OMEGA_M'
    print(param_name)
    param_names = chain.getParamNames()
    par = param_names.parWithName(param_name)
    param_stats = margestats.parWithName(param_name)
    print("Average:", param_stats.mean)

    idx_map = np.argmax(chain.loglikes)
    s8_index_in_samples = chain.index[param_name]
    S8_map = samples[idx_map, s8_index_in_samples]
    print("MAP:", S8_map)

    S8_maxlike = samples[-1, s8_index_in_samples]
    print("Max Likelihood:", S8_maxlike)


In [ ]:
s8_samples = samples[:, s8_index_in_samples]

In [ ]:
np.average(s8_samples, weights=chain.weights)

In [ ]:
chain.weights

In [ ]:
chains=[]
mask_labels = []

for root in roots:

    try:
        samples = np.loadtxt(root_dir + '{}/{}/samples_{}_cell.txt'.format(root,root,root))
        print(len(samples))
        if 'nautilus' in root:
            samples = np.column_stack((np.exp(samples[:,-3]),samples[:,-1]-samples[:,-2],samples[:,0:-3]))
        else:
            samples = np.column_stack((samples[:,-1],samples[:,-2],samples[:,0:-4]))
        np.savetxt(root_dir + '{}/{}/getdist_{}_cell.txt'.format(root,root,root), samples)
        chain = g.samples_for_root(root_dir + '{}/{}/getdist_{}_cell'.format(root,root,root),
                                cache=False,
                                settings={'ignore_rows':0,
                                            'smooth_scale_2D':0.5,
                                            'smooth_scale_1D':0.5})

        chains.append(chain)

        chain = g.samples_for_root(root_dir + '{}/{}/getdist_{}_cell'.format(root,root,root),
                                cache=False,
                                settings={'ignore_rows':0,
                                            'smooth_scale_2D':0.1,
                                            'smooth_scale_1D':0.1})

        chains.append(chain)
        mask_labels.append(1)
    except Exception as e:
        print(e)
        mask_labels.append(0)

In [ ]:
#Plot only cosmological parameters
legend_labels = [
        label
        for i in range(index, index + n_contours)
        for label in (
            rf"$C_\ell$, GLASS mock {i} (smooth 0.5)",
            rf"$C_\ell$, GLASS mock {i} (smooth 0.1)"
        )
]

g.triangle_plot(
    chains,
    ['OMEGA_M','s_8_input', 'SIGMA_8', 'a'],
    legend_labels=legend_labels,
    legend_loc='upper right',
    filled=True,
    markers=markers
)


plt.show()